# Add text type tasks to LabelStudio

In [1]:
import pandas as pd
import numpy as np
import os
import json
from dotenv import load_dotenv
from label_studio_sdk.client import LabelStudio
import sys

# Load environment variables
load_dotenv()

# Direct the notebook to find scripts in parent directory, since it is sitting one level down
os.chdir("..")

sys.path.append(os.getcwd())

# Import the new task classes
from adt_labelstudio.text_type import TextTypeTask
from adt_labelstudio.utils import get_project_annotations, get_ls_project_id_from_name

# Utility functions for normalization (if not already available from utils)
from adt_eval.utils.transcript_cleaner import standardize_transcript, normalize_transcript


In [2]:

# Connect to the Label Studio API and check the connection
LABEL_STUDIO_URL = os.getenv("LABELSTUDIO_URL")
API_KEY = os.getenv("LABELSTUDIO_KEY")

ls_client = LabelStudio(base_url=LABEL_STUDIO_URL, api_key=API_KEY)

text_type_task = TextTypeTask()

In [3]:
file_dir = "output/eval_feb_1/logs/text_extraction/"
file_list = os.listdir(file_dir)

source_project_name = "A1: Text Extraction"
target_project_name = "TEST: Text type" # Adding to test project so as not to overwrite existing annotations


# Get annotations from previous task, used to populate this one
gs_annotations = get_project_annotations(ls_client, project_name=source_project_name)

# Get annotations that have already been done for this task
target_project_tasks = get_project_annotations(ls_client, project_name=target_project_name)

tasks_to_load = []

for f in file_list:

    llm_log, book_name, page_id = text_type_task.get_llm_log(f"{file_dir}/{f}")

    # Confirm that task does not already exist
    if target_project_tasks.shape[0]>0 and (book_name, page_id) in [xy for xy in zip(target_project_tasks['book_id'], target_project_tasks['page_id'])]:
        print(f"Task for {book_name} page {page_id} already exists in LabelStudio and was not added.")
        continue

    print(f"Processing task for {book_name} page {page_id}.")

    # Read in the LLM log file to dataframe
    llm_df = text_type_task.load_llm_log_to_df(llm_log)

    # Get the Gold Standard data as dataframe
    gs_annotation = text_type_task.get_single_annotation(gs_annotations, book_name, page_id)
    gs_df = text_type_task.load_gs_annotation_to_df(gs_annotation)

    # Merge Gold Standard and LLM dataframes onexact match
    matched_df = text_type_task.merge_gs_with_llm(gs_df, llm_df)

    # Create labelstudio task, consisting of input data and predictions
    task_data = text_type_task.populate_task_data(gs_annotation)
    task_predictions = text_type_task.populate_task_predictions(matched_df)
    task_json = text_type_task.create_one_task(task_data, task_predictions)

    tasks_to_load.append(task_json)

# Add the whole list to LabelStudio
if len(tasks_to_load) > 0:
    target_project_id  = get_ls_project_id_from_name(ls_client, project_name=target_project_name)
    ls_client.projects.import_tasks(
                id=target_project_id,
                request=tasks_to_load,
            )
with open("tasks_to_load.json", "w") as f:                  
    json.dump(tasks_to_load, f)

Task for B-54 page 1 already exists in LabelStudio and was not added.
Task for B-54 page 3 already exists in LabelStudio and was not added.
Task for B-89 page 3 already exists in LabelStudio and was not added.
Task for B-89 page 7 already exists in LabelStudio and was not added.
Task for B-89 page 9 already exists in LabelStudio and was not added.
Task for B-89 page 14 already exists in LabelStudio and was not added.
Task for B-89 page 16 already exists in LabelStudio and was not added.
Task for B-89 page 17 already exists in LabelStudio and was not added.
Task for B-89 page 23 already exists in LabelStudio and was not added.
Task for B-89 page 25 already exists in LabelStudio and was not added.
Task for B-89 page 30 already exists in LabelStudio and was not added.
Task for B-89 page 32 already exists in LabelStudio and was not added.
Task for B-89 page 51 already exists in LabelStudio and was not added.
Task for B-89 page 55 already exists in LabelStudio and was not added.
Task for B-